# Data Preparation
This notebook mirrors `pipeline.data_preparation`. It follows the CRISP-DM
data-preparation tasks: select data, clean data, construct data, integrate
data, and format data.



In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pipeline").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.data_preparation import prepare_modeling_data

OUTPUT_DIR = PROJECT_ROOT / "data/processed/modeling_v0.1"
RUN_PREPARATION = False

if RUN_PREPARATION:
    prepare_modeling_data(
        raw_path=PROJECT_ROOT / "enriched_prs_raw.jsonl",
        output_dir=OUTPUT_DIR,
        config_path=PROJECT_ROOT / "config.yaml",
        token_cap=1000,
        batch_size=32,
    )



## Select Data
The raw enriched pull request records are split by repository. The modeling
target is `review_concern`; rows without review activity are excluded from
train/validation/test matrices.



In [2]:
summary = json.loads(
    (OUTPUT_DIR / "dataset_modeling_v0.1.preparation_summary.json").read_text()
)
manifest = json.loads(
    (OUTPUT_DIR / "dataset_modeling_v0.1.feature_manifest.json").read_text()
)
summary["split_summary"]



{'all': {'rows': 7563,
  'repos': 4758,
  'review_concern_0': 407,
  'review_concern_1': 7149,
  'accepted_quality_rows': 721},
 'train': {'rows': 6063,
  'repos': 3802,
  'review_concern_0': 327,
  'review_concern_1': 5736,
  'accepted_quality_rows': 570},
 'val': {'rows': 717,
  'repos': 475,
  'review_concern_0': 42,
  'review_concern_1': 675,
  'accepted_quality_rows': 65},
 'test': {'rows': 777,
  'repos': 476,
  'review_concern_0': 38,
  'review_concern_1': 738,
  'accepted_quality_rows': 86}}

## Clean Data
Bot, documentation-only, generated/vendor-only, missing source patch, and
linked-issue failures are retained as quality metadata. They are not used as
model features.



In [3]:
pd.DataFrame(summary["top_reject_reasons"], columns=["reason", "rows"])



,reason,rows
0,no_explicit_linked_issue,6639
1,no_source_patches,1449
2,no_source_files,1446
3,not_enough_meaningful_review_comments,494
4,not_enough_discussion,411
5,docs_only,327
6,bot_author,51
7,not_enough_review_comments,43
8,generated_or_vendor_only,31
9,bad_diff_size,6


## Construct Data
The table contains pull-request-intrinsic tabular features plus a 768-dim
ModernBERT embedding of source-code `files[].patch`, capped to 1,000 tokens
per row.



In [4]:
print("target:", manifest["target_column"])
print("feature_count:", len(manifest["feature_columns"]))
print("embedding_columns:", len(manifest["embedding_columns"]))
print("token_cap:", manifest["embedding_token_cap"])



target: review_concern
feature_count: 805
embedding_columns: 768
token_cap: 1000


## Integrate Data
Pull request metadata, linked issue data, changed files, and source patches
are integrated into one row per pull request.



In [5]:
all_df = pd.read_parquet(
    OUTPUT_DIR / "dataset_modeling_v0.1.all.parquet",
    columns=[
        "example_id",
        "repo",
        "pr_number",
        "split",
        "review_concern",
        "accepted_quality",
        "top_language",
        "source_patch_token_count_capped",
    ],
)
all_df.head()



,example_id,repo,pr_number,split,review_concern,accepted_quality,top_language,source_patch_token_count_capped
0,openshift__release__pull_62266,openshift/release,62266,train,1.0,0,unknown,0
1,didx-xyz__acapy-cloud__pull_1391,didx-xyz/acapy-cloud,1391,train,1.0,0,python,1000
2,opendatahub-io__kserve__pull_522,opendatahub-io/kserve,522,train,0.0,0,go,1000
3,openshift__release__pull_62971,openshift/release,62971,train,0.0,0,unknown,0
4,ExtremeFiretop__MerlinAutoUpdate-Router__pull_426,ExtremeFiretop/MerlinAutoUpdate-Router,426,train,0.0,0,unknown,0


## Format Data
The `.npz` files contain numeric `X` and `y` arrays and can be passed directly
to scikit-learn style `.fit()` and `.predict()` calls.



In [6]:
train = np.load(OUTPUT_DIR / "dataset_modeling_v0.1.train.npz")
val = np.load(OUTPUT_DIR / "dataset_modeling_v0.1.val.npz")
print("train:", train["X"].shape, train["y"].shape, np.bincount(train["y"]))
print("val:", val["X"].shape, val["y"].shape, np.bincount(val["y"]))

model = DummyClassifier(strategy="most_frequent")
model.fit(train["X"], train["y"])
pred = model.predict(val["X"])
print("fit_predict_ok:", pred.shape)


train: (6063, 805) (6063,) [ 327 5736]
val: (717, 805) (717,) [ 42 675]
fit_predict_ok: (717,)
